**Leer Bronze**


In [0]:
from pyspark.sql import functions as F

In [0]:
customers_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.customers")
policies_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.policies")
claims_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.claims")
telematics_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.telematics")
training_images_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.training_images")
claim_images_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images")
claim_images_metadata_bz = spark.read.table(f"{CATALOG}.{BRONZE_SCHEMA}.claim_images_metadata")

1.Customers


In [0]:
customers_slv = (
    customers_bz
    .withColumn("birth_date", F.to_date("birth_date"))
    
    # limpiar nombre
    .withColumn("name_clean", F.trim(F.col("name")))
    
    # separar nombre y apellido
    .withColumn("firstname", F.split(F.col("name_clean"), " ").getItem(0))
    .withColumn("lastname", F.split(F.col("name_clean"), " ").getItem(1))
    
    # construir direccion
    .withColumn("address", F.concat_ws(" ", F.col("borough"), F.col("zip_code")))
    
    .dropDuplicates(["customer_id"])
)

2.Policies

In [0]:
policies_slv = (
    policies_bz
    .withColumn("premium", F.col("premium").cast("double"))
    
    # validar campos clave
    .filter(F.col("policy_id").isNotNull())
    .filter(F.col("customer_id").isNotNull())
    
    .dropDuplicates(["policy_id"])
)

3. Claims


In [0]:
claims_slv = (
    claims_bz
    .withColumn("claim_date", F.to_date("claim_date"))
    .withColumn("incident_date", F.to_date("incident_date"))
    .withColumn("license_issue_date", F.to_date("license_issue_date"))
    
    # validar campos importantes
    .filter(F.col("claim_id").isNotNull())
    .filter(F.col("policy_id").isNotNull())
    
    .dropDuplicates(["claim_id"])
)

4.Telematics

In [0]:
telematics_slv = (
    telematics_bz
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    
    # filtrar coordenadas válidas
    .filter(
        (F.col("lat") >= -90) & (F.col("lat") <= 90) &
        (F.col("lon") >= -180) & (F.col("lon") <= 180)
    )
    
    .dropDuplicates()
)

5. Training images

In [0]:
training_images_slv = (
    training_images_bz
    .withColumn("image_name", F.element_at(F.split(F.col("path"), "/"), -1))
    
    # label desde nombre archivo (ej: car_accident_1.jpg → car)
    .withColumn("label", F.split(F.col("image_name"), "_").getItem(0))
)

6. Claim images

In [0]:
claim_images_slv = (
    claim_images_bz
    .withColumn("image_name", F.element_at(F.split(F.col("path"), "/"), -1))
    
    # IMPORTANTE: no inferir claim
)

7.Claim Images Metadata

In [0]:
claim_images_metadata_slv = (
    claim_images_metadata_bz
    .select(
        "image_id",
        "image_name",
        "claim_no",
        "chassis_no"
    )
    .dropDuplicates(["image_id"])
)

8. Guardar Silver

In [0]:
customers_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.customers_clean")

policies_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.policies_clean")

claims_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.claims_clean")

telematics_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.telematics_clean")

training_images_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.training_images")

claim_images_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.claim_images")

claim_images_metadata_slv.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG}.silver.claim_images_metadata_clean")

9. Validaciones

In [0]:
print("Nulos customers:", customers_slv.filter(F.col("customer_id").isNull()).count())

print("Fechas mal parseadas claims:", claims_slv.filter(F.col("claim_date").isNull()).count())

print("Coordenadas inválidas:", telematics_slv.filter(
    (F.col("lat").isNull()) | (F.col("lon").isNull())
).count())